In [34]:
import tensorflow as tf
import os



if 'COLAB_GPU' in os.environ:
    MAIN_DIR_PATH = os.path.join("/", "content")
    LABELS_PATH = os.path.join(MAIN_DIR_PATH, "drive","MyDrive","datasets", "turker_scores_full_interview.csv")
    DRIVE_PATH = os.path.join(MAIN_DIR_PATH, 'drive' ,'MyDrive')
    SAVE_CHECKPOINT_PATH = os.path.join(DRIVE_PATH, 'model_checkpoints', 'ckpt_epoch_{epoch:02d}.keras')
    LOAD_CHECKPOINT_PATH = os.path.join(MAIN_DIR_PATH, "ckpt_epoch_01.keras")
    VIDEOS_FRAMES_PATH = os.path.join(MAIN_DIR_PATH,'dataset', "videos_frames")

    # Enable GPU memory growth to avoid OOM errors
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        tf.config.experimental.set_memory_growth(physical_devices[0], True)
    # Enable XLA (Accelerated Linear Algebra) for potential speedups
    tf.config.optimizer.set_jit(True)

    from google.colab import drive; drive.mount('/content/drive')
    !unzip -q /content/drive/MyDrive/datasets/videos_frames.zip -d /content/dataset/

elif os.path.exists('/kaggle'):
    MAIN_DIR_PATH = os.path.join("/", "kaggle", "working")
    LABELS_PATH= os.path.join(MAIN_DIR_PATH, 'turker_scores_full_interview.csv')
    SAVE_CHECKPOINT_PATH = os.path.join(MAIN_DIR_PATH, 'ckpt_epoch_{epoch:02d}.keras')
    LOAD_CHECKPOINT_PATH = os.path.join(
        "/", "kaggle", "input", "checkpoint", "tensorflow2", "epoch4", "1", "ckpt_epoch_04.keras")
    VIDEOS_FRAMES_PATH = os.path.join(MAIN_DIR_PATH,'dataset', "videos_frames")

    # Enable memory growth
    physical_devices = tf.config.list_physical_devices('GPU')
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    # Enable XLA (Accelerated Linear Algebra)
    tf.config.optimizer.set_jit(True)
    # Set graph optimizations
    tf.config.optimizer.set_experimental_options({
        "layout_optimizer": True,
        "constant_folding": True,
        "shape_optimization": True,
        "remapping": True,
        "arithmetic_optimization": True,
        "dependency_optimization": True,
        "loop_optimization": True,
        "function_optimization": True,
        "debug_stripper": True
    })

    !gdown 1C_YeHVXKrkKOPysJxG6sVMdESKSjaCv_
    !gdown 1e9km9KLY3qkVxlCutDBZnC49263dsd5F
    !unzip -q /kaggle/working/videos_frames.zip -d /kaggle/working/dataset
else:
    from hireverse.utils.utils import BASE_DIR
    MAIN_DIR_PATH =BASE_DIR
    LABELS_PATH= os.path.join(
        BASE_DIR, "data", "external", "turker_scores_full_interview.csv"
    )
    SAVE_CHECKPOINT_PATH = os.path.join(MAIN_DIR_PATH, 'ckpt_epoch_{epoch:02d}.keras')
    LOAD_CHECKPOINT_PATH = os.path.join(
        BASE_DIR, "ckpt_epoch_12.keras")
    VIDEOS_FRAMES_PATH = os.path.join(MAIN_DIR_PATH,'data', 'processed','videos_frames')

In [35]:
import os
# MY_Chosen_LABELS = ['Colleague', 'Engaged', 'Excited', 'EyeContact', 'Smiled', 'Calm']
LABELS = ['Engaged', 'NotStressed', 'Friendly', 'Excited', 'Colleague', 'Focused']
participant_ids =['P1', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P20', 'P21', 'P22', 'P24', 'P25', 'P27', 'P29', 'P30', 'P31', 'P32', 'P33', 'P34', 'P35', 'P37', 'P42', 'P43', 'P44', 'P45', 'P47', 'P48', 'P49', 'P50', 'P52', 'P53', 'P55', 'P56', 'P57', 'P58', 'P59', 'P60', 'P61', 'P62', 'P63', 'P64', 'P65', 'P66', 'P67', 'P69', 'P70', 'P71', 'P72', 'P73', 'P74', 'P76', 'P77', 'P78', 'P79', 'P80', 'P81', 'P83', 'P84', 'P85', 'P86', 'P89', 'PP1', 'PP3', 'PP4', 'PP5', 'PP6', 'PP7', 'PP8', 'PP10', 'PP11', 'PP12', 'PP13', 'PP14', 'PP15', 'PP16', 'PP17', 'PP20', 'PP21', 'PP22', 'PP24', 'PP25', 'PP27', 'PP29', 'PP30', 'PP31', 'PP32', 'PP33', 'PP34', 'PP35', 'PP37', 'PP42', 'PP43', 'PP44', 'PP45', 'PP47', 'PP48', 'PP49', 'PP50', 'PP52', 'PP53', 'PP55', 'PP56', 'PP57', 'PP58', 'PP59', 'PP60', 'PP61', 'PP62', 'PP63', 'PP64', 'PP65', 'PP66', 'PP67', 'PP69', 'PP70', 'PP71', 'PP72', 'PP73', 'PP74', 'PP76', 'PP77', 'PP78', 'PP79', 'PP80', 'PP81', 'PP83', 'PP84', 'PP85', 'PP86', 'PP89']
INPUT_SIZE = (224, 224)
INFERENCE = True

FRAMES_BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 8
TEST_BATCH_SIZE = FRAMES_BATCH_SIZE*TRAIN_BATCH_SIZE
EPOCHS = 60  # 12 continuous hours of T4, 30 hours per week total

In [36]:
import os
import re
from typing import List, Tuple
import pandas as pd
import cv2
import numpy as np
from natsort import natsorted

class DatasetHandler:
    @staticmethod
    def get_labels_dict(participant_id: str):
        df = pd.read_csv(
            LABELS_PATH
        )
        df = df.loc[
            (df["Participant"] == participant_id.lower()) & (df["Worker"] == "AGGR")
        ]
        return df.iloc[0].to_dict()

    @staticmethod
    def get_participant_ids():
        p_participant_numbers, pp_participant_numbers = DatasetHandler._get_p_and_pp_participant_number()
        participant_ids = []
        for prefix, participant_numbers in [
            ("P", p_participant_numbers),
            ("PP", pp_participant_numbers),
        ]:
            for participant_number in participant_numbers:
                participant_id = f"{prefix}{participant_number}"
                participant_ids.append(participant_id)
        return participant_ids

    def get_participant_dir(participant_id):
        return os.path.join(VIDEOS_FRAMES_PATH, participant_id)

    @staticmethod
    def get_sorted_participant_frames_images(participant_id, is_image_greyscale=False) -> List[np.ndarray]:
        participant_dir = DatasetHandler.get_participant_dir(participant_id)
        frames = []
        for filename in natsorted(os.listdir(participant_dir)):
            if any(filename.endswith(ext) for ext in [".jpg", ".jpeg", ".png"]):
                image_path = os.path.join(participant_dir, filename)
                image = cv2.imread(image_path)
                if is_image_greyscale:
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                if image is not None:
                    frames.append(image)
        return frames

    @staticmethod
    def yield_sorted_participant_frames_images(participant_id, convert_to_greyscale=False):
        participant_dir = DatasetHandler.get_participant_dir(participant_id)
        for filename in natsorted(os.listdir(participant_dir)):
            if any(filename.endswith(ext) for ext in [".jpg", ".jpeg", ".png"]):
                image_path = os.path.join(participant_dir, filename)
                image = cv2.imread(image_path)
                if convert_to_greyscale:
                    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                if image is not None:
                    yield image

    @staticmethod
    def get_number_of_frames(participant_id):
        participant_dir = DatasetHandler.get_participant_dir(participant_id)
        return len(
            [
                f
                for f in os.listdir(participant_dir)
                if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ]
        )

In [37]:
# import os
# from PIL import Image
# import numpy as np
# from multiprocessing import Pool, cpu_count

# def preprocess_images_for_mobilenetv2(frames_path, target_size):
#     for filename in os.listdir(frames_path):
#         if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tiff')):
#             filepath = os.path.join(frames_path, filename)
#             try:
#                 with Image.open(filepath) as img:
#                     resized_img = img.resize(target_size, Image.Resampling.LANCZOS)

#                     img_array = np.array(resized_img)
#                     if img_array.ndim == 2:
#                         img_array = np.expand_dims(img_array, axis=-1)
#                     elif img_array.ndim == 3 and img_array.shape[-1] != 1:
#                         raise ValueError(f"Image {filename} is not 1-channel grayscale")

#                     rgb_array = np.repeat(img_array, 3, axis=-1)
#                     final_img = Image.fromarray(rgb_array, 'RGB')
#                     final_img.save(filepath)

#             except Exception as e:
#                 print(f"Error processing {filepath}: {e}")

# def process_subdir(args):
#     subdir, target_size = args
#     preprocess_images_for_mobilenetv2(subdir, target_size)
#     print(f"Finished {subdir[-4:]}")

# if __name__ == "__main__":
#     all_subdirs = []
#     for root, dirs, _ in os.walk(VIDEOS_FRAMES_PATH):
#         all_subdirs.append(root)

#     args_list = [(subdir, INPUT_SIZE) for subdir in all_subdirs]

#     workers = min(cpu_count(), len(all_subdirs))

#     with Pool(workers) as pool:
#         pool.map(process_subdir, args_list)

In [38]:
# import os
# import random

# for participant_id in participant_ids:
#     participant_dir = DatasetHandler.get_participant_dir(participant_id)
#     c = 0
#     num_frames = random.randint(30, 60)

#     for f in os.listdir(participant_dir):
#         c += 1
#         if c > num_frames:
#             file_path = os.path.join(participant_dir, f)
#             os.remove(file_path)

In [39]:
from sklearn.model_selection import train_test_split

train_ids, test_ids = train_test_split(participant_ids[:int(len(participant_ids)/2)], test_size=0.2, random_state=42)

train_ids = natsorted(train_ids + [f"P{pid}" for pid in train_ids])
test_ids = natsorted(test_ids + [f"P{pid}" for pid in test_ids])
# train_gen = train_generator(train_ids)


In [40]:
import gc
import cv2
import numpy as np
from tensorflow.keras.utils import to_categorical
import time
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def participant_frames_generator(participant_id, number_of_frames_in_batch=FRAMES_BATCH_SIZE):
    total_frames = DatasetHandler.get_number_of_frames(participant_id)
    # Truncate to discard partial batches
    total_framesـafter_truncation = (total_frames // number_of_frames_in_batch) * number_of_frames_in_batch

    frame_generator = DatasetHandler.yield_sorted_participant_frames_images(
        participant_id, convert_to_greyscale=False
    )
    label_dict = DatasetHandler.get_labels_dict(participant_id)

    for _ in range(0, total_framesـafter_truncation, number_of_frames_in_batch):
        frames_batch = []
        for __ in range(number_of_frames_in_batch):
            frame = next(frame_generator)
            frame = tf.keras.applications.mobilenet_v2.preprocess_input(frame)   # [-1,1]
            frames_batch.append(frame)
        yield frames_batch, {comp: to_100class(label_dict[comp]) for comp in LABELS}

def from_100class(class_index):
    class_index = max(0, min(class_index, 99))  # Clamp to [0, 99]
    score = (class_index * (9 / 100)) + 1
    return round(score, 2)  # Rounded for readability

def to_100class(score_1_to_10):
    score_clamped = max(1.0, min(score_1_to_10, 10.0))  # Ensure score ∈ [1, 10]
    scaled = (score_clamped - 1) * (100 / 9)  # Scale to [0, 100)
    return min(int(scaled), 99)  # Truncate to 0–99

def train_generator(train_ids, participants_per_batch=TRAIN_BATCH_SIZE):
    while True:
        shuffled_ids = np.random.permutation(train_ids)
        # Initialize generators and track remaining batches per participant
        participant_gens = {}
        batches_left = {}
        for pid in shuffled_ids:
            total_frames = DatasetHandler.get_number_of_frames(pid)
            num_batches = total_frames // FRAMES_BATCH_SIZE
            if num_batches > 0:
                participant_gens[pid] = participant_frames_generator(pid, FRAMES_BATCH_SIZE)
                batches_left[pid] = num_batches
        active_participants = list(batches_left.keys())

        # Generate batches until no participants can form a full batch
        while len(active_participants) > 0:
            selected_pids = np.random.choice(
                active_participants,
                size=participants_per_batch,
                replace=len(active_participants) < participants_per_batch
            )

            X_batch = []
            y_batch = {comp: [] for comp in LABELS}
            exhausted_pids = set()

            for pid in selected_pids:
                if batches_left[pid] <= 0:
                    continue  # Skip if no batches left (due to replacement)
                try:
                    frames, labels = next(participant_gens[pid])
                    X_batch.extend(frames)
                    for comp in LABELS:
                        y_batch[comp].extend([labels[comp]] * len(frames))
                    batches_left[pid] -= 1
                    if batches_left[pid] == 0:
                        exhausted_pids.add(pid)
                except StopIteration:
                    exhausted_pids.add(pid)

            for pid in exhausted_pids:
                if pid in active_participants:
                    active_participants.remove(pid)

            # expected_frames = participants_per_batch * FRAMES_BATCH_SIZE
            # if len(X_batch) == expected_frames:
            X_batch = np.array(X_batch)
            indices_list = np.random.permutation(len(X_batch))
            # print(f"\nyield batch having numbe_r of frames: {len(X_batch)}")
            yield X_batch[indices_list], {comp: np.array(y_batch[comp])[indices_list] for comp in LABELS} # returns a shuffled version of the entire X_batch, not just a single item.
            # elif not active_participants:  # ✅ Only break if no participants left
            #     print(f"\n skipped batch having number of frames: {len(X_batch)}")
            #     break

In [41]:
import tensorflow as tf

policy = tf.keras.mixed_precision.Policy('mixed_float16')
tf.keras.mixed_precision.set_global_policy(policy)

In [42]:
from math import ceil

total_number_of_train_frames = sum(DatasetHandler.get_number_of_frames(pid) for pid in train_ids)
number_of_frame_batches = sum(
    DatasetHandler.get_number_of_frames(pid) // FRAMES_BATCH_SIZE
    for pid in train_ids
)
steps_per_epoch = ceil(number_of_frame_batches / TRAIN_BATCH_SIZE )
discarded_frames = total_number_of_train_frames - steps_per_epoch * FRAMES_BATCH_SIZE* TRAIN_BATCH_SIZE
print(discarded_frames)
print(f"{(100 * discarded_frames / total_number_of_train_frames):.1f}%")

770
0.1%


In [43]:
from tensorflow.keras import layers, models

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

def create_model(input_shape):
    inputs = tf.keras.Input(shape=input_shape)

    # Input preprocessing (MobileNetV2 expects [-1, 1] range)
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

    # Base model
    base_model = MobileNetV2(
        input_tensor=x,
        weights='imagenet',
        include_top=False,
        pooling='avg'  # Better than GlobalAveragePooling
    )

    # Strategic unfreezing (top 20 layers trainable)
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    # ===== CRITICAL BN LAYER ADDITIONS =====
    # 1. After base model
    x = base_model.output
    x = layers.BatchNormalization()(x)  # First BN

    # 2. Before Dense activation
    x = layers.Dense(256, use_bias=False)(x)  # Disable bias since BN will add it
    x = layers.BatchNormalization()(x)  # Second BN
    x = layers.Activation('relu')(x)

    # 3. After Dropout
    x = layers.Dropout(0.3)(x)
    x = layers.BatchNormalization()(x)  # Third BN

    # Output heads with forced float32 for stability
    outputs = [layers.Dense(100, activation='softmax', dtype='float32', name=comp)(x)
               for comp in LABELS]

    model = tf.keras.Model(inputs=inputs, outputs=outputs)

    # Learning rate schedule with warmup
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=1e-5,
        decay_steps=steps_per_epoch*EPOCHS,
        warmup_target=1e-3,
        warmup_steps=steps_per_epoch*3
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr_schedule),
        loss={comp: tf.keras.losses.SparseCategoricalCrossentropy()
              for comp in LABELS},
        metrics=['accuracy' for comp in LABELS]
    )
    return model


height, width = INPUT_SIZE
input_shape = (height, width, 3)
model = create_model(input_shape)
# model.summary()

In [44]:
import tensorflow as tf
import re

callbacks = [
      tf.keras.callbacks.EarlyStopping(
          monitor='loss',
          patience=5,
          min_delta=0.001
      ),
      tf.keras.callbacks.ModelCheckpoint(
          filepath=SAVE_CHECKPOINT_PATH,
          verbose = 1,
          save_weights_only=False,
          save_freq='epoch',
          monitor='loss',
          save_best_only=True
      )
  ]

initial_epoch = 0
if os.path.exists(LOAD_CHECKPOINT_PATH):
    match = re.search(r'epoch(\d+)', LOAD_CHECKPOINT_PATH)
    if match:
        initial_epoch = int(match.group(1))
    model = tf.keras.models.load_model(LOAD_CHECKPOINT_PATH)

if not INFERENCE:
  history = model.fit(
      train_generator(train_ids),
      steps_per_epoch=steps_per_epoch,
      epochs=EPOCHS,
      callbacks = callbacks,
      initial_epoch=initial_epoch,
  )
  model.save('final_model.keras')

In [51]:
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
import numpy as np
from tqdm import tqdm
from collections import defaultdict

video_data = defaultdict(lambda: {'frame_preds': [], 'video_true': None})

for pid in tqdm(test_ids[:5:2], desc="Predicting videos"):
    predictions_per_competency = defaultdict(list)

    for frames_batch, labels in participant_frames_generator(pid, number_of_frames_in_batch=TEST_BATCH_SIZE):
        frames_batch_np = np.array(frames_batch)  # Shape: (batch_size, height, width, 3)
        batch_preds = model.predict(frames_batch_np, verbose=0)  # List of 6 arrays, each (batch_size, 1)

        for i, comp in enumerate(LABELS):
            predictions_per_competency[comp].extend(batch_preds[i].flatten()) 

        video_data[pid]['video_true'] = np.array([labels[comp] for comp in LABELS])


    video_data[pid]['frame_preds'] = np.array([np.mean(predictions_per_competency[comp]) for comp in LABELS])

video_true = []
video_pred = []

for pid in video_data:
    video_true.append(video_data[pid]['video_true'])  
    video_pred.append(video_data[pid]['frame_preds']) 

video_true = np.array(video_true) 
video_pred = np.array(video_pred)  

Predicting videos: 100%|██████████| 3/3 [01:41<00:00, 33.71s/it]


In [61]:
print(video_true)
print(video_pred)
# print(video_data)

[[50 48 47 44 48 53]
 [55 52 50 43 46 57]
 [39 48 34 28 36 52]]
[[0.01 0.01 0.01 0.01 0.01 0.01]
 [0.01 0.01 0.01 0.01 0.01 0.01]
 [0.01 0.01 0.01 0.01 0.01 0.01]]


In [ ]:
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
import numpy as np
from tqdm import tqdm
from collections import defaultdict

video_data = defaultdict(lambda: {'frame_preds': [], 'video_true': None})

for pid in tqdm(test_ids[:], desc="Predicting videos"):
    frame_preds = []
    c = 0

    predictions_per_competency = np.empty((len(LABELS), 0))
    for frames_batch, labels in participant_frames_generator(pid, number_of_frames_in_batch=FRAMES_BATCH_SIZE*TRAIN_BATCH_SIZE):
        frames_batch_np = np.array(frames_batch)  # Convert list of frames to NumPy array
        batch_preds = model.predict(frames_batch_np, verbose=0)
        temp = np.argmax(batch_preds, axis=-1)  # get index of max on last axis (softmax's 100 classes)
        predictions_per_competency = np.concatenate([predictions_per_competency, temp], axis=1)

        c += 1
        if c == 3:
            pass

    video_data[pid]['frame_preds'] = predictions_per_competency
    video_data[pid]['video_true'] = np.array([labels[comp] for comp in LABELS])



video_true = []
video_pred = []

for pid in video_data:
    video_pred.append(np.mean(video_data[pid]['frame_preds'], axis=1))
    video_true.append(video_data[pid]['video_true'])

video_true = np.array(video_true)  # shape: (num_videos, 6)
video_pred = np.array(video_pred)   # shape: (num_videos, 6)

(224, 224, 3)
(1, 224, 224, 3)
1/1 [==============================] - 0s 152ms/step
(6, 1, 100)
Engaged: [0.00658576 0.01316202 0.00760386 0.02958269 0.00420192 0.00370151
 0.0053978  0.00957724 0.00777483 0.00294525 0.00552716 0.00442518
 0.00119191 0.00296589 0.011821   0.00139609 0.00722204 0.01015736
 0.0033934  0.01000393 0.00770539 0.015139   0.01362465 0.00917807
 0.00741369 0.00633126 0.00342659 0.01062501 0.00233481 0.01500718
 0.00731139 0.00844085 0.00594757 0.01368339 0.00804896 0.01770901
 0.0106675  0.00554706 0.00643075 0.01320236 0.00497389 0.02259404
 0.00922497 0.01141137 0.00204506 0.01516784 0.0028686  0.02161972
 0.00461661 0.00500593 0.01032557 0.00809567 0.00262021 0.00412278
 0.02347648 0.01225194 0.02106551 0.00781722 0.00584499 0.00341891
 0.01271187 0.01092595 0.00254586 0.00823787 0.00213372 0.00560151
 0.01254797 0.0138303  0.00621133 0.00836443 0.00717725 0.0036312
 0.01324231 0.00253322 0.0079616  0.01378732 0.0084027  0.00380041
 0.02599649 0.00803172 0.

In [ ]:
from scipy.stats import pearsonr
from sklearn.metrics import r2_score
import numpy as np


for i, comp in enumerate(LABELS):
    true_vals = video_true[:, i]
    pred_vals = video_pred[:, i]

    # Pearson r
    r, _ = pearsonr(true_vals, pred_vals)

    # R^2 score
    r2 = r2_score(true_vals, pred_vals)

    print(f"{comp}: Pearson r = {r:.4f}, R² = {r2:.4f}")